# Stage 1 Earthquake Detection Using a 1D CNN

The 1D convolutional neural network classifies preprocessed three-component seismic waveform windows as earthquake or noise. The model operates directly on waveform samples without manually engineered features.

## Load the full Stage 1 dataset

The same train, validation, and test splits used for the Random Forest experiment are loaded to maintain a consistent evaluation framework across Stage 1 models.

In [2]:
import numpy as np

X_train = np.load(
    "../data/processed/stage1_full_train_X.npy"
)
y_train = np.load(
    "../data/processed/stage1_full_train_y.npy"
)

X_val = np.load(
    "../data/processed/stage1_full_validation_X.npy"
)
y_val = np.load(
    "../data/processed/stage1_full_validation_y.npy"
)

X_test = np.load(
    "../data/processed/stage1_full_test_X.npy"
)
y_test = np.load(
    "../data/processed/stage1_full_test_y.npy"
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (7461, 3, 2000) (7461,)
Validation: (1743, 3, 2000) (1743,)
Test: (1796, 3, 2000) (1796,)


## Class distribution

The class distribution is examined before training because the full Stage 1 dataset contains substantially more earthquake windows than noise windows.

In [3]:
unique, counts = np.unique(y_train, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count}")

Class 0: 700
Class 1: 6761


## Class-weighted training

Class weights are applied to the training loss to account for the imbalance between earthquake and noise windows. The weighting is calculated from the training split only.

In [4]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Class counts from training data
n_noise = np.sum(y_train == 0)
n_earthquake = np.sum(y_train == 1)

# Weight each class inversely to its frequency
weight_noise = len(y_train) / (2 * n_noise)
weight_earthquake = len(y_train) / (2 * n_earthquake)

print(f"Noise weight:       {weight_noise:.4f}")
print(f"Earthquake weight:  {weight_earthquake:.4f}")

Noise weight:       5.3293
Earthquake weight:  0.5518


## CNN architecture

A 1D convolutional neural network is used to learn discriminative patterns directly from the three-component seismic waveform windows.

In [5]:
import torch
import torch.nn as nn


class EarthquakeCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(
                in_channels=3,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=7,
                padding=3
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.classifier(x)


model = EarthquakeCNN()

print(model)

EarthquakeCNN(
  (features): Sequential(
    (0): Conv1d(3, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (7): ReLU()
    (8): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


## Convert waveforms to tensors

The preprocessed waveform arrays are converted to PyTorch tensors for CNN training and evaluation. The original amplitude-preserving waveforms are used as model inputs.

In [6]:
X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print("Train tensors:", X_train_tensor.shape, y_train_tensor.shape)
print("Validation tensors:", X_val_tensor.shape, y_val_tensor.shape)
print("Test tensors:", X_test_tensor.shape, y_test_tensor.shape)

Train tensors: torch.Size([7461, 3, 2000]) torch.Size([7461])
Validation tensors: torch.Size([1743, 3, 2000]) torch.Size([1743])
Test tensors: torch.Size([1796, 3, 2000]) torch.Size([1796])


In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = EarthquakeCNN().to(device)
print("Using device:", device)
print(model)

Using device: cpu
EarthquakeCNN(
  (features): Sequential(
    (0): Conv1d(3, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (4): ReLU()
    (5): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (6): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (7): ReLU()
    (8): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


## Training configuration

The CNN is trained using an unweighted cross-entropy loss, with equal contribution from the two classes. The Adam optimizer is used to update the model parameters.

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


## Data loaders

Mini-batches are created from the training, validation, and test tensors. Training batches are shuffled, while validation and test batches retain their original order.

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 117
Validation batches: 28
Test batches: 29


## CNN training

The CNN is trained for 15 epochs using the training split. Validation loss and F1-score are calculated after each epoch to monitor generalization. The model state with the highest validation F1-score is retained for subsequent threshold selection and final test evaluation.

In [12]:
import copy
from sklearn.metrics import f1_score

num_epochs = 15

best_val_f1 = 0.0
best_model_state = copy.deepcopy(model.state_dict())

train_losses = []
val_losses = []
val_f1_scores = []

for epoch in range(num_epochs):

    # Training
    model.train()

    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_train_loss += (
            loss.item() * X_batch.size(0)
        )

    epoch_train_loss = (
        running_train_loss /
        len(train_loader.dataset)
    )

    # Validation
    model.eval()

    running_val_loss = 0.0

    val_predictions = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_val_loss += (
                loss.item() * X_batch.size(0)
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_targets.extend(
                y_batch.cpu().numpy()
            )

    epoch_val_loss = (
        running_val_loss /
        len(val_loader.dataset)
    )

    epoch_val_f1 = f1_score(
        val_targets,
        val_predictions,
        zero_division=0
    )

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    val_f1_scores.append(epoch_val_f1)

    if epoch_val_f1 > best_val_f1:
        best_val_f1 = epoch_val_f1

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val F1: {epoch_val_f1:.4f}"
    )

Epoch 01/15 | Train Loss: 15.5002 | Val Loss: 0.3296 | Val F1: 0.9550
Epoch 02/15 | Train Loss: 0.3320 | Val Loss: 0.3132 | Val F1: 0.9524
Epoch 03/15 | Train Loss: 0.2929 | Val Loss: 0.2779 | Val F1: 0.9550
Epoch 04/15 | Train Loss: 0.2764 | Val Loss: 0.2637 | Val F1: 0.9547
Epoch 05/15 | Train Loss: 0.2667 | Val Loss: 0.2665 | Val F1: 0.9544
Epoch 06/15 | Train Loss: 0.2635 | Val Loss: 0.2618 | Val F1: 0.9547
Epoch 07/15 | Train Loss: 0.2517 | Val Loss: 0.2567 | Val F1: 0.9547
Epoch 08/15 | Train Loss: 0.2485 | Val Loss: 0.2487 | Val F1: 0.9550
Epoch 09/15 | Train Loss: 0.2398 | Val Loss: 0.2512 | Val F1: 0.9550
Epoch 10/15 | Train Loss: 0.2328 | Val Loss: 0.2655 | Val F1: 0.9550
Epoch 11/15 | Train Loss: 0.2295 | Val Loss: 0.2339 | Val F1: 0.9550
Epoch 12/15 | Train Loss: 0.2159 | Val Loss: 0.2251 | Val F1: 0.9561
Epoch 13/15 | Train Loss: 0.2413 | Val Loss: 0.2355 | Val F1: 0.9547
Epoch 14/15 | Train Loss: 0.2130 | Val Loss: 0.2212 | Val F1: 0.9550
Epoch 15/15 | Train Loss: 0.2009 

## Restore the best CNN checkpoint

The CNN state with the highest validation F1-score is restored before classification threshold selection and final test evaluation.

In [13]:
model.load_state_dict(best_model_state)
model.eval()

print(f"Best validation F1: {best_val_f1:.6f}")

Best validation F1: 0.956130


## Validation earthquake probabilities

Earthquake probabilities are generated for the validation set to determine the classification threshold before evaluating the held-out test set.

In [14]:
with torch.no_grad():
    val_logits = model(
        X_val_tensor.to(device)
    )

val_probabilities = torch.softmax(
    val_logits,
    dim=1
)[:, 1].cpu().numpy()

In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd

thresholds = np.arange(0.10, 1.00, 0.05)

threshold_results = []

for threshold in thresholds:

    y_val_pred = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_val,
            y_val_pred
        ),
        "precision": precision_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            y_val_pred,
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

display(
    threshold_results.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

,threshold,accuracy,precision,recall,f1
12,0.70,0.923121,0.932938,0.986817,0.959121
13,0.75,0.920252,0.955514,0.957313,0.956413
10,0.60,0.916810,0.919467,0.996234,0.956312
8,0.50,0.916236,0.917003,0.998745,0.956130
7,0.45,0.916236,0.917003,0.998745,0.956130
6,0.40,0.915663,0.916475,0.998745,0.955843
9,0.55,0.915663,0.916955,0.998117,0.955816
11,0.65,0.915663,0.921329,0.992467,0.955576
5,0.35,0.914515,0.915420,0.998745,0.955269
0,0.10,0.913941,0.913941,1.000000,0.955036


## CNN classification threshold

A classification threshold of 0.70 is selected based on the highest validation F1-score. The threshold is fixed before evaluation on the held-out test set.

In [18]:
final_cnn_threshold = 0.70

print(f"Selected CNN threshold: {final_cnn_threshold:.2f}")

Selected CNN threshold: 0.70


## Final CNN test evaluation

The best CNN checkpoint and classification threshold selected using the validation set are applied to the held-out test split. No further model or threshold selection is performed using the test set.

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.eval()

with torch.no_grad():
    test_logits = model(
        X_test_tensor.to(device)
    )

test_probabilities = torch.softmax(
    test_logits,
    dim=1
)[:, 1].cpu().numpy()

y_test_pred_cnn = (
    test_probabilities >= final_cnn_threshold
).astype(int)

cnn_test_accuracy = accuracy_score(
    y_test,
    y_test_pred_cnn
)

cnn_test_precision = precision_score(
    y_test,
    y_test_pred_cnn,
    zero_division=0
)

cnn_test_recall = recall_score(
    y_test,
    y_test_pred_cnn,
    zero_division=0
)

cnn_test_f1 = f1_score(
    y_test,
    y_test_pred_cnn,
    zero_division=0
)

cnn_cm = confusion_matrix(
    y_test,
    y_test_pred_cnn
)

print(f"Threshold: {final_cnn_threshold:.2f}")
print(f"Accuracy:  {cnn_test_accuracy:.4f}")
print(f"Precision: {cnn_test_precision:.4f}")
print(f"Recall:    {cnn_test_recall:.4f}")
print(f"F1-score:  {cnn_test_f1:.4f}")

print("\nConfusion matrix:")
print(cnn_cm)

Threshold: 0.70
Accuracy:  0.9304
Precision: 0.9388
Recall:    0.9885
F1-score:  0.9630

Confusion matrix:
[[  44  106]
 [  19 1627]]
